In [1]:
import sys
!{sys.executable} -m pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00a 0:00:01


In [2]:
import os, shutil

BASE = "/kaggle/input/datasets/organizations/gti-upm/leapgestrecog/leapGestRecog"
WORK = "/kaggle/working"

CLASS_MAP = {
    "normal": ["01_palm", "05_thumb", "06_index", "07_ok"],
    "grab":   ["03_fist", "04_fist_moved"],
    "pull":   ["08_palm_moved", "10_down"]
}

for cls, folders in CLASS_MAP.items():
    os.makedirs(f"{WORK}/frames/{cls}", exist_ok=True)

    for subject in os.listdir(BASE):
        subject_path = os.path.join(BASE, subject)
        if not os.path.isdir(subject_path):
            continue

        for gesture in folders:
            g_path = os.path.join(subject_path, gesture)
            if not os.path.exists(g_path):
                continue

            for img in os.listdir(g_path):
                if img.endswith((".png", ".jpg")):
                    src = os.path.join(g_path, img)
                    dst = f"{WORK}/frames/{cls}/{subject}_{gesture}_{img}"
                    shutil.copy(src, dst)

print("✅ Dataset organized")

✅ Dataset organized


In [3]:
import cv2, os
from concurrent.futures import ThreadPoolExecutor

CLASSES = ["normal", "grab", "pull"]
WORK = "/kaggle/working"

def crop_hand(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        padding = 20
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(img.shape[1], x + w + padding)
        y2 = min(img.shape[0], y + h + padding)
        return img[y1:y2, x1:x2]
    else:
        return img

def process_image(args):
    in_path, out_path = args
    img = cv2.imread(in_path)
    if img is None:
        return
    crop = crop_hand(img)
    if crop.size > 0:
        cv2.imwrite(out_path, crop)

for cls in CLASSES:
    in_folder = f"{WORK}/frames/{cls}"
    out_folder = f"{WORK}/hand_frames/{cls}"
    os.makedirs(out_folder, exist_ok=True)
    tasks = [
        (os.path.join(in_folder, f), os.path.join(out_folder, f))
        for f in os.listdir(in_folder)
    ]
    with ThreadPoolExecutor(max_workers=8) as executor:
        executor.map(process_image, tasks)

print("✅ Hand cropping done")

✅ Hand cropping done


In [4]:
import random, shutil

WORK = "/kaggle/working"
SPLIT = 0.8

for cls in CLASSES:
    src = f"{WORK}/hand_frames/{cls}"
    files = os.listdir(src)
    random.shuffle(files)

    split_idx = int(SPLIT * len(files))

    for split_name, split_files in [("train", files[:split_idx]), ("test", files[split_idx:])]:
        dst = f"{WORK}/dataset/{split_name}/{cls}"
        os.makedirs(dst, exist_ok=True)

        for f in split_files:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))

print("✅ Train-test split complete")

✅ Train-test split complete


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

IMG_SIZE = (128,128)
BATCH = 32
EPOCHS = 8
WORK = "/kaggle/working"

train_gen = ImageDataGenerator(rescale=1./255, horizontal_flip=True)
test_gen  = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    f"{WORK}/dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical"
)

test_data = test_gen.flow_from_directory(
    f"{WORK}/dataset/test",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical"
)

print("Class mapping:", train_data.class_indices)

model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.3),

    Dense(3,activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_data, validation_data=test_data, epochs=EPOCHS)

model.save(f"{WORK}/cnn_3class_model.keras")

# Save as .h5
model.save(f"{WORK}/cnn_3class_model.h5")
print("✅ .h5 saved")

# Save as .pth
import torch
from collections import OrderedDict
import numpy as np

state_dict = OrderedDict()
for layer in model.layers:
    weights = layer.get_weights()
    if not weights:
        continue
    if len(weights) >= 1:
        state_dict[f"{layer.name}.weight"] = torch.tensor(np.array(weights[0]))
    if len(weights) >= 2:
        state_dict[f"{layer.name}.bias"] = torch.tensor(np.array(weights[1]))

torch.save(state_dict, f"{WORK}/cnn_3class_model.pth")
print("✅ .pth saved")

print("✅ Model saved")

Found 12800 images belonging to 3 classes.
Found 3200 images belonging to 3 classes.
Class mapping: {'grab': 0, 'normal': 1, 'pull': 2}
Epoch 1/8


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


400/400 ━━━━━━━━━━━━━━━━━━━━ 31s 69ms/step - accuracy: 0.8651 - loss: 0.3196 - val_accuracy: 0.9969 - val_loss: 0.0104
Epoch 2/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.9939 - loss: 0.0178 - val_accuracy: 0.9966 - val_loss: 0.0079
Epoch 3/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.9977 - loss: 0.0096 - val_accuracy: 0.9987 - val_loss: 0.0050
Epoch 4/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.9983 - loss: 0.0054 - val_accuracy: 1.0000 - val_loss: 9.4486e-04
Epoch 5/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.9979 - loss: 0.0044 - val_accuracy: 0.9997 - val_loss: 9.3246e-04
Epoch 6/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.9984 - loss: 0.0039 - val_accuracy: 0.9997 - val_loss: 0.0011
Epoch 7/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 26s 66ms/step - accuracy: 0.9988 - loss: 0.0036 - val_accuracy: 0.9984 - val_loss: 0.0040
Epoch 8/8
400/400 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.9985 - loss: 0.0039 - val_accura

✅ .h5 saved
✅ .pth saved
✅ Model saved


In [12]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
print("✅ Working")

✅ Working


In [13]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from ultralytics import YOLO

WORK = "/kaggle/working"
MODEL_PATH = f"{WORK}/cnn_3class_model.keras"
VIDEO_PATH = "/kaggle/input/datasets/aniketsahu09/chain-snatching-cctv-dataset/Chain_Snatching_Videos/Chain_Snatching06.mp4"
IMG_SIZE = (128,128)
CONFIDENCE_THR = 0.6
LABEL_MAP = {0:"grab",1:"normal",2:"pull"}

model = load_model(MODEL_PATH, compile=False)
yolo = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
print("Video opened:", cap.isOpened())
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 20
print("Video info:", width, height, fps)

out = cv2.VideoWriter(
    f"{WORK}/output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

frame_count = 0

def preprocess(frame):
    img = cv2.resize(frame, IMG_SIZE)
    return np.expand_dims(img/255.0, axis=0)

def predict(frame):
    pred = model.predict(preprocess(frame), verbose=0)[0]
    i = np.argmax(pred)
    return LABEL_MAP[i], pred[i]

while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video")
        break
    frame_count += 1
    results = yolo(frame, classes=[0], verbose=False)
    if results[0].boxes is not None:
        for box in results[0].boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            person = frame[y1:y2, x1:x2]
            if person.size == 0:
                continue
            gray = cv2.cvtColor(person, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                x, y, w, h = cv2.boundingRect(c)
                padding = 20
                x1c = max(0, x - padding)
                y1c = max(0, y - padding)
                x2c = min(person.shape[1], x + w + padding)
                y2c = min(person.shape[0], y + h + padding)
                hand = person[y1c:y2c, x1c:x2c]
            else:
                hand = person
            if hand.size == 0:
                continue
            gesture, conf = predict(hand)
            if conf > CONFIDENCE_THR:
                label = f"{gesture} {round(conf*100,1)}%"
                color = (0,255,0)
                if gesture in ["grab","pull"]:
                    color = (0,0,255)
                cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
                cv2.putText(frame,label,(x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,0.7,color,2)
    out.write(frame)

cap.release()
out.release()
print("Frames written:", frame_count)
print("✅ Video saved → /kaggle/working/output.mp4")

Video opened: True
Video info: 450 360 25.05123845191717
End of video
Frames written: 419
✅ Video saved → /kaggle/working/output.mp4


In [14]:
from IPython.display import Video
Video("/kaggle/working/output.mp4")

In [15]:
import os
os.path.getsize("/kaggle/working/output.mp4")

6010340